<div dir="rtl">

# الدرس 41: الشبكات الالتفافية للصور

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### الالتفاف بيدك، ثم بباي تورش

</div>

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

def conv2d(image, kernel):
    """Slide the kernel over the image: multiply element-wise and sum (no padding, stride 1)."""
    k = kernel.shape[0]
    n = image.shape[0] - k + 1
    out = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            out[i, j] = np.sum(image[i:i + k, j:j + k] * kernel)
    return out

image = np.array([[0, 0, 9, 9, 9]] * 5, dtype=float)
vertical_edge = np.array([[-1, 0, 1]] * 3, dtype=float)
print(conv2d(image, vertical_edge))

# same computation in PyTorch; shapes: (batch, channels, height, width)
t = F.conv2d(torch.tensor(image)[None, None], torch.tensor(vertical_edge)[None, None])
print(t[0, 0])

<div dir="rtl">

### مرشّحات الحواف على صورة

</div>

In [ ]:
yy, xx = np.mgrid[0:40, 0:40]
picture = np.zeros((40, 40))
picture[8:32, 5:17] = 1.0                                  # a rectangle
picture[(yy - 20) ** 2 + (xx - 28) ** 2 < 64] = 1.0        # a disc
horizontal_edge = vertical_edge.T
edges_v = conv2d(picture, vertical_edge)
edges_h = conv2d(picture, horizontal_edge)
print("output shape:", edges_v.shape, " min/max:", edges_v.min(), edges_v.max())

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
for ax, img, title in zip(axes, [picture, edges_v, edges_h],
                          ["image", "vertical-edge filter", "horizontal-edge filter"]):
    ax.imshow(img, cmap="RdBu_r" if img is not picture else "gray_r",
              vmin=-3 if img is not picture else 0, vmax=3 if img is not picture else 1)
    ax.set_title(title, fontsize=10); ax.axis("off")
plt.tight_layout()
plt.show()

<div dir="rtl">

### البيانات: صور الأرقام كلها

</div>

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

X, y = load_digits(return_X_y=True)
X = (X / 16.0).reshape(-1, 1, 8, 8).astype(np.float32)          # (n, channels, h, w)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
X_tr, X_te = torch.tensor(X_tr), torch.tensor(X_te)
y_tr, y_te = torch.tensor(y_tr), torch.tensor(y_te)
print("train:", tuple(X_tr.shape), " test:", tuple(X_te.shape))

<div dir="rtl">

### الشبكتان

</div>

In [ ]:
def make_cnn():
    torch.manual_seed(0)
    return nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(32 * 2 * 2, 10))

def make_mlp():
    torch.manual_seed(0)
    return nn.Sequential(nn.Flatten(), nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 10))

def count(model):
    return sum(p.numel() for p in model.parameters())

x = X_tr[:1]
for layer in make_cnn():
    x = layer(x)
    print(f"{layer.__class__.__name__:10} -> {tuple(x.shape)}")
print("CNN parameters:", count(make_cnn()), "  MLP parameters:", count(make_mlp()))

<div dir="rtl">

### التدريب، واختبار الإزاحة

</div>

In [ ]:
def shift(X, dx=0, dy=0):
    """Move images dx pixels right and dy pixels down, filling the gap with zeros."""
    out = torch.roll(X, shifts=(dy, dx), dims=(2, 3))
    if dx > 0: out[..., :, :dx] = 0
    if dx < 0: out[..., :, dx:] = 0
    if dy > 0: out[..., :dy, :] = 0
    if dy < 0: out[..., dy:, :] = 0
    return out

def train(model, epochs=30, augment=False):
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    loss_fn = nn.CrossEntropyLoss()
    g = torch.Generator().manual_seed(0)
    for _ in range(epochs):
        model.train()
        for idx in torch.randperm(len(X_tr), generator=g).split(32):
            xb = X_tr[idx]
            if augment:                                   # random shift of -1, 0 or +1 pixel
                dx, dy = torch.randint(-1, 2, (2,), generator=g).tolist()
                xb = shift(xb, dx, dy)
            loss = loss_fn(model(xb), y_tr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def accuracy(model, X):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(1) == y_te).float().mean().item()

for name, make in [("MLP", make_mlp), ("CNN", make_cnn)]:
    model = train(make())
    print(f"{name}: test {accuracy(model, X_te):.3f} | shifted 1px {accuracy(model, shift(X_te, 1)):.3f}"
          f" | shifted 2px {accuracy(model, shift(X_te, 2)):.3f}")

<div dir="rtl">

### زيادة البيانات: علّمها الإزاحة

</div>

In [ ]:
for name, make in [("MLP", make_mlp), ("CNN", make_cnn)]:
    model = train(make(), epochs=60, augment=True)
    print(f"{name} + shifts: test {accuracy(model, X_te):.3f} | shifted 1px {accuracy(model, shift(X_te, 1)):.3f}"
          f" | shifted 2px {accuracy(model, shift(X_te, 2)):.3f}")

<div dir="rtl">

## تمرين: احسب طريق الأشكال

شبكة لصور ملونة 32 في 32:

1. التفاف: 3 قنوات إلى 8، مرشّح 5 في 5، بدون حشو، خطوة 1.
2. تجميع بالأكبر 2 في 2.
3. التفاف: 8 قنوات إلى 16، مرشّح 3 في 3، حشو 1، خطوة 1.
4. تجميع بالأكبر 2 في 2.
5. فرد، ثم طبقة كاملة الاتصال إلى 10.

احسب بيدك شكل المخرج بعد كل طبقة، وعدد المعاملات الكلي. ثم تحقق بالكود.

</div>

In [ ]:
# اكتب حلّك هنا
